In [ ]:
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
import os
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# 設定最上層路徑
root_data_path = "/mnt/hdd/KuroSiwo_data/KuroSiwo/data/"
root_dir = Path(root_data_path)

# 基礎輸出路徑
base_out_dir = "/home/chunen/nas/bigdata/final/kurosiwo_S1_DEM/"

# 取得 root 底下所有的事件資料夾 (例如 118, 130, 147...)
event_dirs = sorted([d for d in root_dir.iterdir() if d.is_dir()])

print(f"總共找到 {len(event_dirs)} 個事件資料夾: {[d.name for d in event_dirs]}")

# 依序處理每一個事件資料夾
for event_dir in event_dirs:
    print(f"\n{'='*60}")
    print(f" 🚀 開始處理事件資料夾: {event_dir.name} ")
    print(f"{'='*60}")
    
    # 建立該事件專屬的輸出資料夾，例如 .../merge/118/
    event_out_dir = os.path.join(base_out_dir, event_dir.name)
    os.makedirs(event_out_dir, exist_ok=True)
    
    # 取得這個事件底下所有的子資料夾 (排除 00 資料夾)
    sub_dirs = sorted([d for d in event_dir.iterdir() if d.is_dir() and d.name != "00"])
    
    if not sub_dirs:
        print(f"事件資料夾 {event_dir.name} 下沒有找到需要處理的子資料夾！跳過。")
        continue

    # 依序處理每個子資料夾 (例如 01, 02, 03 ...)
    for sub_dir in sub_dirs:
        print(f"\n{'-'*40}")
        print(f" 📁 進入子資料夾: {sub_dir.name} ")
        print(f"{'-'*40}")
        
        # 找到底下所有的 tif 檔案
        tif_files = list(sub_dir.rglob("*.tif"))
        if not tif_files:
            print(f"在 {sub_dir.name} 中沒有找到任何 tif 檔案，跳過。")
            continue
        
        # 根據檔名前兩個單詞分組
        groups = defaultdict(list)
        for fp in tif_files:
            parts = fp.name.split("_")
            if len(parts) >= 2:
                prefix = f"{parts[0]}_{parts[1]}"
            else:
                # 萬一檔名沒有底線，則用主檔名
                prefix = fp.stem 
                
            groups[prefix].append(fp)
            
        print(f"✅ 在 {sub_dir.name} 中共找到 {len(tif_files)} 個檔案，分為 {len(groups)} 種前綴。")
        
        # 輸出檔名前綴：{事件資料夾}_{子資料夾} (例如 118_01)
        parent_name = event_dir.name
        current_name = sub_dir.name
        name_prefix = f"{parent_name}_{current_name}" # "118_01"
        
        # 分別處理每個前綴的影像群組
        for prefix, files in groups.items():
            print(f"\n  ➤ 處理前綴 [{prefix}] -> 共 {len(files)} 個檔案")
            
            src_files = []
            # 開啟這組的影像
            for fp in files:
                try:
                    src = rasterio.open(str(fp))
                    src_files.append(src)
                except rasterio.errors.RasterioIOError as e:
                    print(f"  ⚠️ 警告：無法讀取檔案，已跳過 -> {fp}")
                    
            if not src_files:
                print(f"  [{prefix}] 沒有成功讀取任何影像，跳過。")
                continue
                
            print(f"  ⏳ 開始拼接 [{prefix}] ...")
            # 拼接
            mosaic, out_transform = merge(src_files)
            
            # 複製 metadata（以第一張為基準）
            out_meta = src_files[0].meta.copy()
            
            # 更新 metadata (設定為 COG 所需的區塊與壓縮)
            out_meta.update({
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": out_transform,
                "tiled": True,          # 啟用區塊
                "blockxsize": 256,      # 常見標準設定為 256
                "blockysize": 256,
                "compress": "DEFLATE"   # 壓縮方式
            })
            
            # 取得日期 (取群組中第一張影像檔名的最後一節)
            date_str = files[0].stem.split("_")[-1]
            
            # 輸出檔名：(例如) 118_01_MS1_IVV_20150203.tif
            out_filename = f"{name_prefix}_{prefix}_{date_str}.tif"
            
            # 檔案輸出到專屬的事件資料夾
            out_filepath = os.path.join(event_out_dir, out_filename)
            
            # 輸出主影像資料
            with rasterio.open(out_filepath, "w", **out_meta) as dest:
                dest.write(mosaic)
                
            # 為了符合 COG，開啟檔案並加入 Overviews（金字塔）
            with rasterio.open(out_filepath, "r+") as dest:
                factors = [2, 4, 8, 16]
                dest.build_overviews(factors, Resampling.average) # 使用 average
                dest.update_tags(ns='rio_overview', resampling='average')
                
            # 關閉檔案
            for src in src_files:
                src.close()
                
            print(f"  ✅ 完成儲存 COG: {out_filename}")

print("\n🎉 全部事件資料夾處理完成！")